
## Heat Equation Part D  
  
# <span style="color:blue">LU factorization and matrix exponentials </span>
---

## Learning objectives

By the end of this part, you will be able to:
- Explain why implicit time-stepping methods require solving linear systems.
- Use LU factorization to efficiently solve repeated linear systems.
- Understand the matrix exponential solution
$$
\mathbf u(𝑡) = e^{tA}\mathbf u_0.
$$
- Compare exact evolution with explicit and implicit time stepping.
- Interpret the heat equation solution in terms of eigenmode decay.


## Contents
#### 1. From PDE to linear ODE system (recap)
#### 2. Implicit Euler as a linear system
#### 3. Implicit Euler with LU factorization
#### 4. The matrix exponential (exact solution)
#### 5. Calculation of the exact solution of the semi-discrete system
#### 6. Explicit Euler for comparison
#### 7. Comparison of methods
#### 8. Interpretation
#### 9. Summary



👉 Advanced methods allow us to solve the heat equation more efficiently by exploiting matrix structure.



## 1. From PDE to linear ODE system (recap)

After spatial discretization (Parts B–C), the heat equation reduces to a linear ODE system
$$
\frac{d\mathbf u}{dt} = A\mathbf u,
$$
where the system matrix $A$ encodes the discrete Laplacian operator and boundary conditions.  Matrix $A$ includes also the thermal diffusivity coefficient $\alpha$, which is set to 1 here.


👉 LU factorization decomposes the system into simpler components that are easier to solve.



## 2. Implicit Euler as a linear system

The backward (implicit) Euler method updates the solution via
$$
(I - \Delta t A)\mathbf u^{n+1} = \mathbf u^n.
$$

Unlike explicit Euler, this scheme is unconditionally stable, but **requires solving a linear system at every timestep.**

### Why LU factorization matters

The matrix $M = I - \Delta t A$ is constant in time. We can factor it once,
$$
M = LU,
$$
and **reuse the factorization at every timestep. Each update then costs only two triangular solves**.

Without LU factorization, each `solve()` would require a fresh factorization, costing $𝑂(𝑁^3)$ per step.
(For this 1D tridiagonal system, specialized solvers can do better, but the LU idea generalizes to higher dimensions and more complex matrices.)

With LU factorization:
- Factorization: $𝑂(𝑁^3)$ **once**
- Each `solve()`: $𝑂(𝑁^2)$ 
- Total cost: dramatically reduced 

This is exactly why implicit PDE solvers almost always use LU, Cholesky, or iterative solvers.

(An example of the use of LU decomposition can be found on YouTube, for example\
<u><a href="https://www.youtube.com/watch?v=t7Nitn8zvt4">
Wrath of Math: LU-Decomposition Method for Solving Linear Systems
</a></u>)

#### Dense vs sparse matrix #### 

We are currently building a dense matrix. 
- The discrete Laplacian is sparse (tridiagonal).
- In realistic PDE problems, sparse LU or iterative solvers are used.
- For tridiagonal matrices, Thomas algorithm gives  $𝑂(𝑁)$ solve cost.


✅ Matrix factorization transforms repeated solves into efficient operations.


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import lu_factor, lu_solve, expm


In [ ]:
# Build the discrete Laplacian matrix A (Dirichlet BCs )

N = 50
L = 1.0
h = L / (N + 1)

main_diag = -2.0 * np.ones(N)
off_diag = 1.0 * np.ones(N - 1)

A = (1 / h**2) * (
    np.diag(main_diag) +
    np.diag(off_diag, 1) +
    np.diag(off_diag, -1)
)


In [ ]:
# Initial condition: Heat distribution in the rod

x = np.linspace(h, L - h, N)
u0 = np.sin(np.pi * x)



## 3. Implicit Euler with LU factorization

This is the part where the time‑stepping of the heat equation actually happens.


In [ ]:
# Parameter setting

dt = 0.01
T = 0.1
steps = int(T / dt)

# 1. Build implicit Euler matrix  M = I - dtA  
M = np.eye(N) - dt * A       

# 2. Precompute the LU decomposition of matrix M:  M = LU
lu, piv = lu_factor(M)

# 3. Solve one linear system per time step
u_imp = u0.copy()
for _ in range(steps):
    u_imp = lu_solve((lu, piv), u_imp)


#### Explanation of the code
- $M$ is the matrix you must invert at every time step to advance the solution.
- $M = LU:  L$ is lower triangular, $U$ is upper triangular, `piv` stores row permutations for numerical stability.
- Because $M$  does not change during the simulation, factoring it once makes each time step extremely fast.
- Loop with `lu_solve`: Solve one linear system per time step

In the time stepping loop above each iteration performs:
$$
 \mathbf u^{𝑛+1} = 𝑀^{−1}\mathbf u^𝑛
$$

But instead of computing an inverse (which is expensive and unstable), you solve:
$$
𝐿𝑈\mathbf u^{𝑛+1} =\mathbf u^𝑛
$$
using forward and backward substitution via `lu_solve`. This is much faster than refactoring or inverting the matrix $M$ at every step.

This is the standard and efficient way to implement the implicit Euler method for PDEs.




## 4. The matrix exponential (exact solution)

### What the matrix exponential is
The matrix exponential is a way to extend the ordinary exponential function 
$𝑒^𝑥$  to square matrices.
For a square matrix $𝐴$, the matrix exponential is defined by the power series:

$$
e^{A}  =  I + A + \frac{A^{2}}{2!} + \frac{A^{3}}{3!} + \cdots = \sum_{k=0}^{\infty} \frac{A^{k}}{k!}
$$
This series always converges for any square matrix, which makes the definition robust.

### Why we care about it
The matrix exponential shows up everywhere in applied mathematics. Solving systems of linear differential equations
$$
\frac{d \mathbf x}{dt} = A \mathbf x(t) \; \; \; \Rightarrow \; \;  \; \mathbf x(t) = e^{tA}\mathbf x(0),
$$

which is the exact solution of the semi-discrete system. This provides a reference solution against which numerical time-stepping schemes can be compared

Matrix exponentials are used, for example: in quantum mechanics (time evolution operators), control theory (state transition matrices), numerical analysis and scientific computing. 


### What it means intuitively

Think of $𝐴$  as describing how a system changes. Then $e^{tA}$ describes how the system evolves over time $𝑡$.

- If $𝐴$ were just a number $a$, then $e^{tA} = e^{ta}$ 

- If $𝐴$  is a matrix, the exponential captures all the interactions between components.



👉 **Matrix exponentials** provide a direct way to **describe time evolution**.


---





 ## 5. Calculation of the exact solution of the semi-discrete system 
 
 From the theory of linear differential equations, the system
$$
\frac{d \mathbf u}{dt} = A \mathbf u
$$
has the **exact solution**
$$
\mathbf u (t) = e^{tA} \mathbf u_0 .
$$
 
 This system is called the **semi-discrete** formulation because:
- space is discrete (grid points, finite differences), 
- time is still continuous.
  
Therefore the heat equation PDE becomes a system of ordinary differential equations.


✅ **Linear algebra** offers multiple complementary ways to **understand** and **compute** the solution.


In [ ]:

# Exact (matrix exponential)
u_exact = expm(T * A) @ u0



## 6. Explicit Euler for comparison


In [ ]:

dt_exp = 0.4* h**2           # stability-limited timestep
steps_exp = int(T / dt_exp)

u_exp = u0.copy()
for _ in range(steps_exp):
    u_exp = u_exp + dt_exp * (A @ u_exp)

print("Timestep:", dt_exp)


## 7. Comparison of methods


In [ ]:

plt.figure()
plt.plot(x, u_exact, label="Exact (matrix exponential)")
plt.plot(x, u_imp, "--", label="Implicit Euler (LU)")
plt.plot(x, u_exp, ".", label="Explicit Euler")
plt.legend()
plt.xlabel("x")
plt.ylabel("u(x, T)")
plt.title("Heat equation time evolution comparison")
plt.show()



## 8. Interpretation

- Our initial condition $ \mathbf u_0 = sin(\pi x) $ is exactly the first eigenvector of the discrete Laplacian, so the sinusoidal   shape does not change during cooling. Only the amplitude decays exponentially.
- Since the time T=0.1 is very short, about 37% of the original amplitude 1.0 is still left (note the scale). That is far from zero — so the sine wave is still very visible.
- **Explicit Euler** approximates $e^{tA}$ via a truncated Taylor expansion and is stability-limited.(Here timestep is 0.0001538)
- **Implicit Euler** Implicit Euler is unconditionally stable (A-stable) for this linear diffusion problem, \
   though it is only first-order accurate in time. (Here timestep is 0.01)
- **Matrix exponential** gives the exact decay of all eigenmodes.
  For Dirichlet Laplacian:
$$
\lambda_k = -\frac{4}{h^2}\,\sin^2\!\left(\frac{k\pi}{2(N+1)}\right)
$$


- The eigenvalues of $A$ are negative, leading to exponential decay of high-frequency modes.



## 9. Summary

- Time evolution after spatial discretization is a linear-algebra problem.
- LU factorization enables efficient implicit time stepping.
- The matrix exponential defines the exact solution and a natural benchmark.

This completes the transition from PDE discretization to linear-system time evolution.


*Heikki Miettinen 2026*